# **CRAWLING, PREPROCESSING, DAN INDEKS**

In [1]:
# -----------------------------------------
# Crawling abstracts from PTA Trunojoyo (Manajemen)
# Copy-paste ke Google Colab dan jalankan
# -----------------------------------------

# (1) install dependencies (jalankan sekali jika perlu)
!pip install -q beautifulsoup4 requests pandas lxml tqdm openpyxl

# (2) imports
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time, random, re
import pandas as pd
from tqdm import tqdm

# (3) konfigurasi
BASE = "https://pta.trunojoyo.ac.id"
DEPT_PATH = "/c_search/byprod/7"   # halaman jurusan Manajemen
START_URL = urljoin(BASE, DEPT_PATH)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}
MAX_PAGES = 200        # batasi jika perlu (naikkan kalau ingin semua halaman)
MAX_DOCS = None        # None = ambil semua; atau set angka mis. 200 untuk uji coba
SLEEP_MIN, SLEEP_MAX = 1.0, 2.0   # jeda antar-request (dalam detik)

session = requests.Session()
session.headers.update(HEADERS)

# (4) helper: ambil semua link detail dari satu page listing
def extract_detail_links_from_listing(html):
    soup = BeautifulSoup(html, "lxml")
    links = []
    for a in soup.find_all("a", href=True):
        if "/welcome/detail/" in a["href"]:
            full = urljoin(BASE, a["href"])
            links.append(full)
    # unikkan sambil menjaga urutan
    return list(dict.fromkeys(links))

# (5) helper: ekstrak abstrak dari halaman detail (coba cari label "Abstrak/Abstraksi")
def extract_info_from_detail(html):
    soup = BeautifulSoup(html, "lxml")
    text_all = soup.get_text("\n").strip()
    # 1) try regex: ambil teks antara "Abstrak/Abstraksi" dan "Abstrac/Abstraction" atau "Kata kunci"
    m = re.search(r'(?:Abstrak|Abstraksi)\s*[:\n\r]*(.*?)(?=(?:Abstrac|Abstraction|Abstrack|Kata kunci|Keywords|$))', text_all, re.I|re.S)
    abstr_ind = m.group(1).strip() if m else ""

    # 2) ambil judul & penulis sederhana dari area atas halaman (fallback)
    lines = [ln.strip() for ln in text_all.splitlines() if ln.strip()]
    title = ""
    author = ""
    # cari "Detail Karya Ilmiah" lalu ambil baris berikutnya sbg title
    for i,ln in enumerate(lines):
        if "Detail Karya Ilmiah" in ln:
            if i+1 < len(lines):
                title = lines[i+1]
            # cari 'Penulis' di beberapa baris ke depan
            for j in range(i+1, min(i+10, len(lines))):
                if lines[j].lower().startswith("penulis"):
                    parts = lines[j].split(":",1)
                    author = parts[1].strip() if len(parts)>1 else ""
                    break
            break
    # fallback jika belum ditemukan
    if not title and lines:
        title = lines[0]
    return {"title": title, "author": author, "abstract_ind": abstr_ind}

# (6) crawl listing pages -> kumpulkan semua link detail
detail_urls = []
print("Mengumpulkan link detail dari listing Manajemen...")
for page in range(1, MAX_PAGES+1):
    if page == 1:
        url = START_URL
    else:
        url = f"{START_URL}/{page}"
    try:
        r = session.get(url, timeout=20)
        if r.status_code != 200:
            print(f"Halaman {page}: status {r.status_code} — hentikan.")
            break
    except Exception as e:
        print(f"Error mengakses {url}: {e}")
        break

    new_links = extract_detail_links_from_listing(r.text)
    if not new_links:
        print(f"Tidak ada link di halaman {page} — kemungkinan sudah habis.")
        break

    # tambahkan link unik baru
    added = 0
    for l in new_links:
        if l not in detail_urls:
            detail_urls.append(l)
            added += 1
    print(f"Halaman {page}: ditemukan {len(new_links)} link, ditambahkan {added} link baru. Total link: {len(detail_urls)}")

    # hentikan kalau sudah mencapai max doc
    if MAX_DOCS and len(detail_urls) >= MAX_DOCS:
        detail_urls = detail_urls[:MAX_DOCS]
        break

    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

print(f"Selesai mengumpulkan. Total detail URLs: {len(detail_urls)}")

# (7) ambil isi dari tiap detail page (ekstraksi abstrak)
rows = []
print("Mengunduh dan mengekstrak tiap detail...")
for u in tqdm(detail_urls):
    try:
        r = session.get(u, timeout=20)
        if r.status_code == 200:
            info = extract_info_from_detail(r.text)
            info["url"] = u
            rows.append(info)
        else:
            print(f"Warning: {u} -> status {r.status_code}")
    except Exception as e:
        print(f"Error ambil {u}: {e}")
    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

# (8) simpan hasil ke CSV / Excel
df = pd.DataFrame(rows)
df.index.name = "id"
csv_path = "/content/pta_manajemen_abstracts.csv"
xlsx_path = "/content/pta_manajemen_abstracts.xlsx"
df.to_csv(csv_path, index=True)
df.to_excel(xlsx_path, index=True)
print(f"Hasil disimpan: {csv_path}  dan  {xlsx_path}")
print(df.head(10).T)  # ringkasan kecil


Mengumpulkan link detail dari listing Manajemen...
Halaman 1: ditemukan 5 link, ditambahkan 5 link baru. Total link: 5
Halaman 2: ditemukan 5 link, ditambahkan 5 link baru. Total link: 10
Halaman 3: ditemukan 5 link, ditambahkan 5 link baru. Total link: 15
Halaman 4: ditemukan 5 link, ditambahkan 5 link baru. Total link: 20
Halaman 5: ditemukan 5 link, ditambahkan 5 link baru. Total link: 25
Halaman 6: ditemukan 5 link, ditambahkan 5 link baru. Total link: 30
Halaman 7: ditemukan 5 link, ditambahkan 5 link baru. Total link: 35
Halaman 8: ditemukan 5 link, ditambahkan 5 link baru. Total link: 40
Halaman 9: ditemukan 5 link, ditambahkan 5 link baru. Total link: 45
Halaman 10: ditemukan 5 link, ditambahkan 5 link baru. Total link: 50
Halaman 11: ditemukan 5 link, ditambahkan 5 link baru. Total link: 55
Halaman 12: ditemukan 5 link, ditambahkan 5 link baru. Total link: 60
Halaman 13: ditemukan 5 link, ditambahkan 5 link baru. Total link: 65
Halaman 14: ditemukan 5 link, ditambahkan 5 link 

100%|██████████| 1000/1000 [40:33<00:00,  2.43s/it]


Hasil disimpan: /content/pta_manajemen_abstracts.csv  dan  /content/pta_manajemen_abstracts.xlsx
id                                                            0  \
title         PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...   
author                                                  SATIYAH   
abstract_ind  si\n\n\n\n\n                                  ...   
url           https://pta.trunojoyo.ac.id/welcome/detail/080...   

id                                                            1  \
title         ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...   
author                                                  Faishal   
abstract_ind  si\n\n\n\n\nTujuan penelitian ini adalah untuk...   
url           https://pta.trunojoyo.ac.id/welcome/detail/090...   

id                                                            2  \
title         PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...   
author                                          Wahyu Kurniawan   
abstract_ind                  

In [1]:
# -----------------------------------------
# Preprocessing Abstrak PTA Trunojoyo (Prodi Manajemen)
# -----------------------------------------

# 1. Install & update library
!pip install --upgrade nltk textblob
!pip install -q Sastrawi pandas openpyxl pyspellchecker

# 2. Import dan download resource NLTK
import nltk
nltk.download('punkt')
nltk.download('stopwords')

import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from spellchecker import SpellChecker
from collections import Counter

# 3. Load data hasil crawling
# Ganti path kalau filenya beda nama/lokasi
df = pd.read_csv("/content/pta_manajemen_abstracts.csv")
print("Jumlah dokumen:", len(df))

# 4. Setup tools
factory = StemmerFactory()
stemmer = factory.create_stemmer()
spell = SpellChecker(language=None)   # opsional, bisa tambahkan kosakata bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

def preprocess_text(text):
    if not isinstance(text, str):
        return []

    # Case folding
    text = text.lower()

    # Hilangkan angka & karakter non-huruf
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Tokenisasi
    tokens = nltk.word_tokenize(text)

    # Stopword removal
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    # Cek ejaan (opsional, bisa aktifkan kalau perlu - agak lambat)
    # tokens = [spell.correction(w) if spell.correction(w) else w for w in tokens]

    # Stemming
    tokens = [stemmer.stem(w) for w in tokens]

    return tokens

# 5. Preprocess semua abstrak
all_tokens = []
df["tokens"] = df["abstract_ind"].apply(lambda x: preprocess_text(x))

for tlist in df["tokens"]:
    all_tokens.extend(tlist)

print("\nContoh token dari 1 dokumen:")
print(df["tokens"].iloc[0][:30])

# 6. Hitung frekuensi kata
freq = Counter(all_tokens)
freq_sorted = freq.most_common()

# 7. Simpan hasil frekuensi
freq_df = pd.DataFrame(freq_sorted, columns=["term", "frequency"])
freq_csv = "/content/freq_terms.csv"
freq_xlsx = "/content/freq_terms.xlsx"
freq_df.to_csv(freq_csv, index=False)
freq_df.to_excel(freq_xlsx, index=False)

print("\nTop 20 kata paling sering muncul:")
print(freq_df.head(20))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Jumlah dokumen: 1000

Contoh token dari 1 dokumen:
['abstrak', 'satiyah', 'pengaruh', 'faktor', 'faktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'bawah', 'bimbing', 'dra', 'anugrahini', 'irawati', 'helm', 'buyung', 'aulia', 'upaya', 'tingkat', 'produktivitas', 'kerja', 'mudah', 'salah', 'usaha', 'produktivitas', 'tingkat']

Top 20 kata paling sering muncul:
          term  frequency
0     pengaruh       5370
1        kerja       4575
2       teliti       4251
3     variabel       3486
4        usaha       2434
5   signifikan       2342
6          uji       2319
7     karyawan       1996
8        nilai       1806
9        hasil       1749
10    analisis       1394
11     positif       1216
12      sampel       1152
13        data       1046
14        tuju       1000
15     parsial        983
16    simultan        936
17      metode        924
18       putus        907
19     tingkat        880


In [2]:
# Tampilkan 5 dokumen pertama beserta token hasil preprocessing
print(df[["title", "tokens"]].head())


                                               title  \
0  PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...   
1  ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...   
2  PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...   
3  Pengukuran Website Quality Pada Situs Sistem A...   
4  PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...   

                                              tokens  
0  [abstrak, satiyah, pengaruh, faktor, faktor, l...  
1  [tuju, teliti, persepsi, brand, association, l...  
2                                                 []  
3  [aplikasi, nyata, manfaat, teknologi, informas...  
4  [abstrak, teliti, metode, kuantitatif, tekan, ...  


In [3]:
# Abstrak pertama sebelum dan sesudah preprocessing
print("Abstrak asli:\n", df["abstract_ind"].iloc[0])
print("\nHasil preprocessing (tokens):\n", df["tokens"].iloc[0])


Abstrak asli:
 si




                                    ABSTRAK
Satiyah, Pengaruh Faktor-faktor Pelatihan dan Pengembangan Terhadap Produktivitas Kerja Dinas Kelautan dan Perikanan Bangkalan. Dibawah bimbingan Dra.Hj.S.Anugrahini Irawati,MM dan  Helmi Buyung Aulia,S,ST.SE,M.MT
Dalam upaya meningkatkan produktivitas kerja tidaklah mudah, oleh karena itu salah satu usaha agar produktivitas meningkat adalah menerapkan program pelatihan dan pengembangan sumber daya manusia (SDM) perlu dilaksanakan dalam instansi agar produktivitas yang tinggi dapat tercapai dengan meningkat kemampuan pegawai agar dapat bekerja secara efektif dan efisien. Dengan adanya pelatihan dan pengembnagan diharapkan pegawai juga mampu menyesuaikan diri dengan kebutuhan-kebutuhan baru atas sikap, tingkah laku keterampilan dan pengetahuan sesuai dengan tuntutan perubahan. Dengan adanya pelatihan dan pengembangan pegawai yang baik dapat mendukung terciptanya suasana kerja yang kondusif dalam instansi, sehingga dengan 

In [4]:
# Simpan hasil preprocessing (abstrak asli + tokens) ke file
prep_csv = "/content/preprocessing_result.csv"
prep_xlsx = "/content/preprocessing_result.xlsx"

df.to_csv(prep_csv, index=False)
df.to_excel(prep_xlsx, index=False)

print(f"Hasil preprocessing disimpan ke:\n{prep_csv}\n{prep_xlsx}")


Hasil preprocessing disimpan ke:
/content/preprocessing_result.csv
/content/preprocessing_result.xlsx
